In [1]:
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [3]:
df = pd.read_stata("dag-sim-data.dta")

print(df.head())

   female   age  exercise  healthy_diet  fast_food        bmi  blood_pressure  \
0     0.0  29.0  1.131711           1.0        1.0  24.443949      129.795074   
1     1.0  21.0  3.251391           1.0        0.0  19.750277      117.414703   
2     1.0  37.0  8.091176           1.0        0.0  20.014652      121.276260   
3     0.0  20.0  4.532768           1.0        0.0  21.793289      118.445755   
4     1.0  25.0  2.411630           1.0        0.0  22.935638      130.366241   

   genetic_risk  heart_disease  
0           0.0            1.0  
1           1.0            1.0  
2           0.0            1.0  
3           0.0            1.0  
4           1.0            1.0  


In [7]:
unique_vals = df['heart_disease'].unique()
print("Unique values:", unique_vals)
print("Number of unique values:", len(unique_vals))

Unique values: [1. 0.]
Number of unique values: 2


In [11]:
# OLS regression 
model_ols = smf.ols("heart_disease ~ fast_food + exercise + healthy_diet + age + bmi + blood_pressure + genetic_risk + female",
                    data=df).fit()

print(model_ols.summary())


                            OLS Regression Results                            
Dep. Variable:          heart_disease   R-squared:                       0.021
Model:                            OLS   Adj. R-squared:                  0.020
Method:                 Least Squares   F-statistic:                     26.36
Date:                Fri, 07 Mar 2025   Prob (F-statistic):           9.18e-41
Time:                        09:41:21   Log-Likelihood:                 13068.
No. Observations:               10000   AIC:                        -2.612e+04
Df Residuals:                    9991   BIC:                        -2.605e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept          0.9597      0.027     35.

In [13]:
#Logit regression 
y = df['heart_disease']
X = df[['fast_food', 'exercise', 'healthy_diet', 'age', 'bmi', 'blood_pressure', 'genetic_risk', 'female']]

X = sm.add_constant(X)

model = sm.Logit(y, X)
result = model.fit_regularized(alpha=1.0, L1_wt=0)

print(result.summary())

C:\Users\SHAYAN\anaconda3\Lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
C:\Users\SHAYAN\anaconda3\Lib\site-packages\statsmodels\discrete\discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))


Optimization terminated successfully    (Exit mode 0)
            Current function value: 0.014823941201365019
            Iterations: 122
            Function evaluations: 133
            Gradient evaluations: 122
                           Logit Regression Results                           
Dep. Variable:          heart_disease   No. Observations:                10000
Model:                          Logit   Df Residuals:                     9994
Method:                           MLE   Df Model:                            5
Date:                Fri, 07 Mar 2025   Pseudo R-squ.:                  0.4912
Time:                        09:41:46   Log-Likelihood:                -143.82
converged:                       True   LL-Null:                       -282.65
Covariance Type:            nonrobust   LLR p-value:                 6.308e-58
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------

In [29]:



print("Columns in dataset:", df.columns)

required_cols = ['heart_disease', 'fast_food', 'exercise', 'age', 'genetic_risk', 'female']
df_clean = df.dropna(subset=required_cols).copy()

df_clean.drop_duplicates(inplace=True)

X_vars = ['fast_food', 'exercise', 'age', 'genetic_risk', 'female']
y_var = 'heart_disease'

df_model = df_clean[X_vars + [y_var]].copy()

numeric_cols = ['fast_food', 'exercise', 'age', 'genetic_risk']
for col in numeric_cols:
    lower_q = df_model[col].quantile(0.01)
    upper_q = df_model[col].quantile(0.99)
    df_model = df_model[(df_model[col] >= lower_q) & (df_model[col] <= upper_q)]

X = df_model[X_vars]
y = df_model[y_var]

X = sm.add_constant(X)

logit_model = sm.Logit(y, X)
result = logit_model.fit(maxiter=500) 
print(result.summary())


Columns in dataset: Index(['female', 'age', 'exercise', 'healthy_diet', 'fast_food', 'bmi',
       'blood_pressure', 'genetic_risk', 'heart_disease'],
      dtype='object')
         Current function value: 0.011010
         Iterations: 500
                           Logit Regression Results                           
Dep. Variable:          heart_disease   No. Observations:                 9652
Model:                          Logit   Df Residuals:                     9646
Method:                           MLE   Df Model:                            5
Date:                Thu, 06 Mar 2025   Pseudo R-squ.:                  0.4452
Time:                        18:17:38   Log-Likelihood:                -106.27
converged:                      False   LL-Null:                       -191.56
Covariance Type:            nonrobust   LLR p-value:                 5.511e-35
                   coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------

C:\Users\SHAYAN\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
